In [ ]:
# Set your API key
from dotenv import load_dotenv
from nnsight import CONFIG
from nnsight.modeling.language import LanguageModel
import os
load_dotenv()
CONFIG.set_default_api_key(os.environ["NDIF_API_KEY"])
CONFIG.API.HOST = "https://api.ndif.us"
CONFIG.save()
# Load model: We'll never actually load the parameters so no need to specify a device_map.
model = LanguageModel("openai-community/gpt2")

/opt/miniconda3/envs/new-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
clean_prompt = "After John and Mary went to the store, Mary gave a bottle of milk to"
corrupted_prompt = (
    "After John and Mary went to the store, John gave a bottle of milk to"
)

In [3]:
correct_index = model.tokenizer(" John")["input_ids"][0] # includes a space
incorrect_index = model.tokenizer(" Mary")["input_ids"][0] # includes a space

print(f"' John': {correct_index}")
print(f"' Mary': {incorrect_index}")

' John': 1757
' Mary': 5335


In [4]:
tokenized_prompt = model.tokenizer(clean_prompt)["input_ids"]
print(model.tokenizer.decode(tokenized_prompt))

After John and Mary went to the store, Mary gave a bottle of milk to


In [5]:
N_LAYERS = len(model.transformer.h)

ioi_patching_results = []
with model.trace() as tracer:
    # define barriers for each layer and token
    barriers_per_layer = [tracer.barrier(len(tokenized_prompt)+1) for _ in range(N_LAYERS)]

    # Clean run
    with tracer.invoke(clean_prompt) as invoker:
        clean_tokens = invoker.inputs[1]["input_ids"][0].save()

        # Get hidden states of all layers in the network.
        # We index the output at 0 because it's a tuple where the first index is the hidden state.
        for layer_idx in range(N_LAYERS):
            hidden_state = model.transformer.h[layer_idx].output[0]

            # call barrier so nnsight can prepare the hidden states for the patching invoke
            barriers_per_layer[layer_idx]()

        # Get logits from the lm_head.
        clean_logits = model.lm_head.output

        # Calculate the difference between the correct answer and incorrect answer for the clean run and save it.
        clean_logit_diff = (
            clean_logits[0, -1, correct_index] - clean_logits[0, -1, incorrect_index]
        ).save()

    # Corrupted run
    with tracer.invoke(corrupted_prompt) as invoker:
        corrupted_logits = model.lm_head.output

        # Calculate the difference between the correct answer and incorrect answer for the corrupted run and save it.
        corrupted_logit_diff = (
            corrupted_logits[0, -1, correct_index]
            - corrupted_logits[0, -1, incorrect_index]
        ).save()


    # Activation Patching Intervention – across all layers & token positions
    for layer_idx in range(len(model.transformer.h)):
        _ioi_patching_results = []

        for token_idx in range(len(tokenized_prompt)):
            # Patching corrupted run at given layer and token

            with tracer.invoke(corrupted_prompt) as invoker:
                # call barrier so nnsight can grab the hidden states for this invoke
                barriers_per_layer[layer_idx]()

                # Patch in the clean hidden states over the corrupted hidden states.
                model.transformer.h[layer_idx].output[0][:, token_idx, :] = hidden_state[:, token_idx, :]

                patched_logits = model.lm_head.output

                patched_logit_diff = (
                    patched_logits[0, -1, correct_index]
                    - patched_logits[0, -1, incorrect_index]
                )

                # Calculate the improvement in the correct token after patching.
                patched_result = (patched_logit_diff - corrupted_logit_diff) / (
                    clean_logit_diff - corrupted_logit_diff
                )

                _ioi_patching_results.append(patched_result.item())

        ioi_patching_results.append(_ioi_patching_results)
        ioi_patching_results.save()

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


In [6]:
import plotly.express as px
def plot_ioi_patching_results(ioi_patching_results,
                              x_labels,
                              plot_title="Normalized Logit Difference After Patching Residual Stream on the IOI Task"):

    fig = px.imshow(
        ioi_patching_results,
        color_continuous_midpoint=0.0,
        color_continuous_scale="RdBu",
        labels={"x": "Position", "y": "Layer","color":"Norm. Logit Diff"},
        x=x_labels,
        title=plot_title,
    )

    return fig

In [7]:
print(f"Clean logit difference: {clean_logit_diff:.3f}")
print(f"Corrupted logit difference: {corrupted_logit_diff:.3f}")

clean_decoded_tokens = [model.tokenizer.decode(token) for token in clean_tokens]
token_labels = [f"{token}_{index}" for index, token in enumerate(clean_decoded_tokens)]

fig = plot_ioi_patching_results(ioi_patching_results,token_labels,"Patching GPT-2-small Residual Stream on IOI task")
fig.show()

Clean logit difference: 4.124
Corrupted logit difference: -2.272
